# Phase 1 independent validation

This notebook verifies Phase 1 behavior and saved evidence; it does not produce aligned temperature data or general statistics. Run all cells in a fresh kernel with the current Phase 1 producer outputs and producer record, project configuration and raw files, and current Phase 2/3 artifacts and validation receipts present. Run from the repository root or `scripts/` so the code can locate `config/project_config.yaml`; arbitrary working directories are rejected. The checks combine temporary synthetic source/audit fixtures, saved-file identities and regression values, and two isolated fresh processes. They do not rerun Phase 1 on all raw data or the Phase 3 clustering grid.

**Receipt side effect:** the first code cell writes `artifacts/phase1_operational/phase1_validation.json` with status `in_progress` before any test executes. It writes through a same-directory temporary file, flush/fsync, and atomic replacement, with temporary-file cleanup. If execution stops before the final runner, this in-progress receipt can remain. The final runner publishes `passed` only after all tests complete without skips and the before/after tested hashes match; a caught exception publishes `failed` and is re-raised.

The validation receipt is the only production-area file this notebook owns. It is separate from producer-owned `phase1_run.json`; producer and downstream receipts/artifacts are read and checked, not regenerated. Fixture reports live under temporary directories, and ordinary checks do not rewrite aligned data or Phase 2/3 scientific outputs. The receipt path is rejected if it traverses a symbolic link. Its tested hashes include both Phase 1 notebooks and all raw-input file bytes, so documentation edits make the prior receipt stale and full raw-tree hashing adds runtime.

In [ ]:
from pathlib import Path
import hashlib, json, os, sys, tempfile, traceback, unittest

ROOT = Path.cwd().resolve()
while not (ROOT / "config" / "project_config.yaml").is_file():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Run from the repository root or scripts directory.")
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
__file__ = str(ROOT / "scripts" / "phase1_test.ipynb")
RECEIPT = ROOT / "artifacts" / "phase1_operational" / "phase1_validation.json"
if RECEIPT.resolve() != RECEIPT or RECEIPT.parent.resolve() != RECEIPT.parent:
    raise ValueError("Phase 1 receipt path must not use a symbolic link.")

def atomic_receipt(record):
    """
    Atomically publish a JSON receipt to the fixed Phase 1 validation path. Write to
    a same-directory temporary file, flush and fsync it, then replace the receipt;
    remove a leftover temporary file on failure and propagate the error.
    """
    RECEIPT.parent.mkdir(parents=True, exist_ok=True)
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=RECEIPT.parent, delete=False) as stream:
            temporary = Path(stream.name)
            json.dump(record, stream, indent=2, allow_nan=False)
            stream.write("\n")
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, RECEIPT)
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()

# This write precedes every test; interruption before the final runner can leave it in progress.
atomic_receipt({"schema_version": 1, "status": "in_progress"})


## Transformation and source-discovery fixture regressions

These six local tests build small files under temporary directories; they do not edit the repository raw inputs. They check case-insensitive exclusion of the explicitly named `temperature_raw.csv` before a file limit, timestamp/name ordering and first-wins duplicate behavior, exclusion metadata in an isolated producer run, empty exclusion-list serialization, independence from monitoring configuration, explicit failed status after an injected write error, and refusal of smoke destinations that overlap protected paths.

In [ ]:
"""Tests for Phase 1 source exclusion and its audit metadata."""

import contextlib
import csv
import io
import json
import tempfile
import unittest
from pathlib import Path

import yaml
from unittest import mock

from scripts.shared_monitoring_engine import load_phase1_definitions

phase1 = load_phase1_definitions(Path(__file__).resolve().parents[1])


class Phase1SourceDiscoveryTests(unittest.TestCase):
    """
    Exercise source discovery, first-wins deduplication, exclusion metadata, and
    smoke-run protections using temporary files. These fixtures do not modify the
    repository raw inputs.
    """
    def setUp(self):
        """
        Create an isolated temporary root and raw-data directory for each test, and
        register cleanup.
        """
        self.temporary = tempfile.TemporaryDirectory()
        self.addCleanup(self.temporary.cleanup)
        self.root = Path(self.temporary.name)
        self.data = self.root / "raw"
        self.data.mkdir()

    def source(self, name, values=(1,)):
        """
        Write a small Temperature CSV under the test raw directory and return its
        path; values create independent fixture rows.
        """
        path = self.data / name
        with path.open("w", newline="", encoding="utf-8") as stream:
            writer = csv.writer(stream)
            writer.writerow(["Time", "DeviceId", "Sensor", "Value"])
            writer.writerows((120, "A", "Temperature", value) for value in values)
        return path

    def test_only_explicit_non_source_is_excluded_before_limit(self):
        """
        Check case-insensitive exclusion of temperature_raw.csv before the positive
        file limit, while timestamped and undated sources retain deterministic
        discovery order and directories are ignored.
        """
        self.source("temperature_raw.csv")
        self.source("TEMPERATURE_RAW.CSV")
        undated = self.source("other.csv")
        later = self.source("2022_01_01_01-00-00_data.csv")
        earlier = self.source("2022_01_01_00-00-00_data.csv")
        (self.data / "folder.csv").mkdir()
        self.assertEqual(
            phase1.discover_csv_files(self.data, None), [undated, earlier, later]
        )
        self.assertEqual(phase1.discover_csv_files(self.data, 1), [undated])
        self.assertEqual(phase1.discover_csv_files(self.data, 0), [])

    def test_first_wins_deduplication_is_unchanged(self):
        """
        Check that sorted source order plus row order keeps the first exact
        time/device/sensor key, skips later duplicates, and records the duplicate
        count in QC.
        """
        self.source("temperature_raw.csv", (999,))
        first = self.source("2022_01_01_00-00-00_data.csv", (10, 20))
        second = self.source("2022_01_01_01-00-00_data.csv", (30,))
        index, aligned, qc = phase1.build_phase1_temperature(
            files=phase1.discover_csv_files(self.data, None),
            bucket_size_seconds=60, stale_threshold_seconds=120,
            min_devices_per_snapshot=1, drop_snapshot_if_stale_over_threshold=True,
            completeness_weight=0.7, freshness_weight=0.3,
        )
        self.assertEqual([row["file_name"] for row in index], [first.name, second.name])
        self.assertEqual(qc["rows_temperature"], 3)
        self.assertEqual(qc["rows_temperature_valid"], 1)
        self.assertEqual(qc["rows_temperature_duplicate"], 2)
        self.assertEqual(aligned[0]["A_temperature"], 10)

    def test_main_records_exclusion_in_run_and_isolated_files(self):
        """
        Run the producer with temporary configuration and destinations, then verify
        the excluded aggregate name is recorded in QC/run metadata and the isolated
        files are emitted.
        """
        self.source("temperature_raw.csv", (999,))
        self.source("2022_01_01_00-00-00_data.csv", (10,))
        repository = Path(__file__).resolve().parents[1]
        config = phase1.load_yaml_config(repository / "config" / "project_config.yaml")
        config["global"]["min_devices_per_snapshot"] = 1
        config_path = self.root / "config.yaml"
        config_path.write_text(yaml.safe_dump(config), encoding="utf-8")
        summary = phase1.run_phase1(repository, config_path=config_path, data_dir=self.data,
                                    limit_files=1, output_dir=self.root / "isolated")
        self.assertEqual(summary["status"], "passed")
        self.assertTrue(summary["quality_gates_passed"])
        self.assertEqual(summary["qc"]["rows_temperature"], 1)
        self.assertEqual(summary["qc"]["excluded_non_source_csv_filenames"], ["temperature_raw.csv"])
        self.assertEqual(json.loads((self.root / "isolated" / "phase1_run.json").read_text()), summary)
        self.assertEqual(set(summary["artifacts"]), {"raw_index", "phase1_aligned"})
        with Path(summary["artifacts"]["phase1_aligned"]).open(newline="") as stream:
            self.assertEqual(next(csv.DictReader(stream))["A_temperature"], "10.0")

    def test_empty_exclusion_list_round_trips_without_mutating_qc(self):
        """
        Check that an empty exclusion list is serialized as JSON in the QC CSV and
        that writing the report does not mutate the input QC mapping.
        """
        qc = {"files_processed": 1, "excluded_non_source_csv_filenames": []}
        report = self.root / "qc.csv"
        phase1.write_qc_report(report, qc)
        self.assertEqual(qc["excluded_non_source_csv_filenames"], [])
        with report.open(newline="") as stream:
            self.assertEqual(
                json.loads(next(csv.DictReader(stream))["excluded_non_source_csv_filenames"]),
                [],
            )

    def test_full_general_run_is_independent_of_monitoring_and_failure_is_explicit(self):
        """
        Check that general Phase 1 can run with only its own configuration and
        outputs, then inject a writer error and verify the producer publishes failed
        status and re-raises without replacing the prior aligned file.
        """
        repository = Path(__file__).resolve().parents[1]
        self.source("2022_01_01_00-00-00_data.csv", (10,))
        config = phase1.load_yaml_config(repository / "config" / "project_config.yaml")
        config["global"]["min_devices_per_snapshot"] = 1
        config["artifacts"] = {
            key: str(self.root / f"{key}.csv") for key in ("raw_index", "phase1_aligned")
        }
        config_path = self.root / "transformation.yaml"
        config_path.write_text(yaml.safe_dump(config))
        (self.root / "scripts").mkdir()
        (self.root / "scripts" / "phase1_implement.ipynb").write_bytes(
            (repository / "scripts" / "phase1_implement.ipynb").read_bytes()
        )
        result = phase1.run_phase1(self.root, config_path=config_path, data_dir=self.data)
        self.assertEqual(result["status"], "passed")
        self.assertFalse((self.root / "config" / "project_config.yaml").exists())
        self.assertEqual([row["series_name"] for row in result["general_statistics"]], ["all_devices_observed"])
        original_aligned = Path(result["artifacts"]["phase1_aligned"]).read_bytes()
        with mock.patch.object(phase1, "write_csv_rows", side_effect=OSError("injected disk failure")):
            with self.assertRaisesRegex(OSError, "injected disk"):
                phase1.run_phase1(self.root, config_path=config_path, data_dir=self.data)
        status_path = self.root / "artifacts" / "phase1_operational" / "phase1_run.json"
        self.assertEqual(json.loads(status_path.read_text())["status"], "failed")
        self.assertEqual(Path(result["artifacts"]["phase1_aligned"]).read_bytes(), original_aligned)

    def test_smoke_refuses_production_and_protected_destinations(self):
        """
        Check that a limited run requires an explicit output directory and rejects
        destinations overlapping production, source, configuration, or archive
        paths.
        """
        repository = Path(__file__).resolve().parents[1]
        with self.assertRaisesRegex(ValueError, "separate explicit"):
            phase1.run_phase1(repository, limit_files=1)
        for folder in ("scripts", "config", "Data", "artifacts", "artifacts\\phase1_outputs",
                       "artifacts\\phase1_operational", "artifacts\\phase1_archive",
                       "artifacts\\phase3_outputs"):
            with self.subTest(folder=folder), self.assertRaisesRegex(ValueError, "overlap"):
                phase1.run_phase1(repository, limit_files=1, output_dir=repository / folder)


## Duplicate-audit fixture regressions

These 16 local tests exercise the optional audit on temporary CSV populations. They distinguish duplicate keys from excess duplicate rows; within-file from cross-file attribution; identical values from conflicting values; and the first Phase 1-kept value from later occurrences. They also cover empty populations, missing inputs, invalid example limits, exact floating-point precision and labels, non-finite values in strict JSON, capped examples, and CLI calls from another working directory.

Exclusion-mode fixtures compare aggregate and timestamped-source **record multisets**, including multiplicity and unmatched occurrences in both directions. Equal matched totals alone are not equality evidence; invalid records and NaN can prevent a complete-verification claim. Spread exceedances use strict `>` boundaries and non-overlapping bins. The audit reports both strict raw-rate-below-threshold and Phase 1's rounded-six-decimal `<=` gate, so a slightly-above raw rate can pass after rounding. Source-only reports use a separate directory and do not replace aggregate-inclusive reports.

In [ ]:
"""Focused tests for the standalone raw Phase 1 duplicate audit."""

import contextlib
import csv
import io
import json
import math
import subprocess
import sys
import tempfile
import unittest
from pathlib import Path

from scripts.shared_monitoring_engine import load_phase1_definitions

REPOSITORY = Path(__file__).resolve().parents[1]
phase1 = load_phase1_definitions(REPOSITORY)
audit_module = load_phase1_definitions(REPOSITORY, audit=True)
SPREAD_THRESHOLDS = audit_module.SPREAD_THRESHOLDS
discover_csv_files = audit_module.discover_csv_files
duplicate_gate = audit_module.duplicate_gate
run_qc = audit_module.run_qc



class DuplicateQCTests(unittest.TestCase):
    """
    Check the optional duplicate-audit API with temporary CSV inputs and reports.
    The audit and its fixtures are separate from ordinary Phase 1 production and
    never use the repository raw files.
    """
    def setUp(self):
        """
        Create temporary source and report directories for an audit fixture and
        register automatic cleanup.
        """
        self.temporary = tempfile.TemporaryDirectory()
        self.addCleanup(self.temporary.cleanup)
        self.root = Path(self.temporary.name)
        self.data = self.root / "raw"
        self.data.mkdir()
        self.output = self.root / "reports"

    def write_source(self, name, rows, header=None):
        """
        Write one CSV fixture with the requested rows and optional header, returning
        the created source path.
        """
        path = self.data / name
        with path.open("w", encoding="utf-8", newline="") as stream:
            writer = csv.writer(stream)
            writer.writerow(header or ["Time", "DeviceId", "Sensor", "Value"])
            writer.writerows(rows)
        return path

    def audit(self, limit=100):
        """
        Run aggregate-inclusive run_qc against the fixture directory with a
        configurable conflict-example limit and return its summary and examples.
        """
        with contextlib.redirect_stderr(io.StringIO()):
            result = run_qc(self.data, self.output, limit)
        with (self.output / "qc_phase1_duplicate_conflicts.csv").open(
            encoding="utf-8", newline=""
        ) as stream:
            examples = list(csv.DictReader(stream))
        with (self.output / "qc_phase1_duplicate_summary.json").open(
            encoding="utf-8"
        ) as stream:
            self.assertEqual(json.load(stream), result)
        return result, examples

    def test_metrics_provenance_order_and_phase1_parity(self):
        """
        Check duplicate-key versus excess-row metrics, same-file/cross-file
        attribution, first-kept Phase 1 value provenance, deterministic ordering,
        example caps, and parity with Phase 1 duplicate handling.
        """
        first = self.write_source(
            "2022_02_11_13-45-52_data.csv",
            [
                (120, "A", "Temperature", "20"),
                ("120.9", "A", "Temperature", "20.0"),
                (180, "A", "Temperature", 10),
                (240, "A", "Temperature", 1),
                (240, "A", "Temperature", 3),
                (300, "A", "Temperature", 5),
                (360, "A", "Temperature", 0),
                (360, "A", "Temperature", "-0.0"),
                (180, "B", "Temperature", 99),
                (120, "A", "Humidity", 40),
                ("bad", "A", "Temperature", 1),
                (400, "", "Temperature", 1),
                (400, "A", "Temperature", "bad"),
                (400, "A", "temperature", 1),
            ],
        )
        second = self.write_source(
            "2022_02_11_14-42-55_data.csv",
            [
                (120, "A", "Temperature", 20),
                (180, "A", "Temperature", 14),
                (300, "A", "Temperature", "5.0000000005"),
            ],
        )
        # Schema-invalid rows are excluded even when Sensor is Temperature.
        self.write_source("broken.csv", [(500, "Temperature", 9)], ["Time", "Sensor", "Value"])
        self.assertEqual(
            discover_csv_files(self.data), phase1.discover_csv_files(self.data, None)
        )
        summary, examples = self.audit(limit=2)
        _, _, baseline = phase1.build_phase1_temperature(
            files=phase1.discover_csv_files(self.data, None),
            bucket_size_seconds=60, stale_threshold_seconds=60,
            min_devices_per_snapshot=1,
            drop_snapshot_if_stale_over_threshold=True,
            completeness_weight=0.5, freshness_weight=0.5,
        )
        for field in ("rows_total", "rows_schema_valid", "rows_schema_invalid", "rows_temperature"):
            self.assertEqual(summary[field], baseline[field], field)
        self.assertEqual(summary["duplicate_rows"], baseline["rows_temperature_duplicate"])
        self.assertEqual(summary["unique_keys"], baseline["rows_temperature_valid"])
        self.assertEqual(summary["rows_temperature"], 12)
        self.assertEqual(summary["rows_temperature_raw"], 16)
        self.assertEqual(summary["rows_temperature_invalid"], 4)
        self.assertEqual(summary["unique_keys"], 6)
        self.assertEqual(summary["duplicate_rows"], 6)
        self.assertEqual(summary["duplicate_key_count"], 5)
        self.assertEqual(summary["duplicate_multiplicity_distribution"], {"2": 4, "3": 1})
        self.assertEqual(summary["identical_value_duplicate_keys"], 2)
        self.assertEqual(summary["identical_value_duplicate_keys_percent"], 40)
        self.assertEqual(summary["conflicting_value_duplicate_keys"], 3)
        self.assertEqual(summary["conflicting_value_duplicate_keys_percent"], 60)
        self.assertEqual(summary["conflicting_value_difference_max"], 4)
        self.assertAlmostEqual(summary["conflicting_value_difference_mean"], (6 + 5e-10) / 3)
        self.assertEqual(summary["conflicting_keys_with_spread_at_most_1e_9"], 1)
        self.assertEqual(summary["cross_file_duplicate_keys"], 3)
        self.assertEqual(summary["cross_file_identical_value_duplicate_keys"], 1)
        self.assertEqual(summary["within_file_only_duplicate_keys"], 2)
        self.assertEqual(summary["mixed_cross_and_within_file_duplicate_keys"], 1)
        self.assertEqual(summary["cross_file_excess_rows"], 3)
        self.assertEqual(summary["within_file_excess_rows"], 3)
        self.assertTrue(summary["primarily_cross_file_by_key"])
        self.assertFalse(summary["primarily_cross_file_by_excess_rows"])
        self.assertEqual(summary["top_20_source_file_pairs"], [{
            "source_files": [first.name, second.name], "shared_duplicate_keys": 3,
        }])
        self.assertEqual([row["time_epoch"] for row in examples], ["180", "240"])
        self.assertEqual(examples[0]["kept_value_phase1"], "10.0")
        self.assertEqual(examples[0]["kept_source_file"], first.name)
        self.assertEqual(examples[0]["kept_row_number"], "4")
        sources = json.loads(examples[0]["source_files"])
        self.assertEqual(sources[0]["values"], [10])
        self.assertEqual(sources[1]["values"], [14])
        self.assertEqual(summary["conflict_examples_written"], 2)

    def test_no_duplicates_and_zero_example_limit(self):
        """
        Check zero duplicate counts and undefined percentages/spreads for a
        no-duplicate population, and verify a zero example limit returns no
        examples.
        """
        self.write_source("raw.csv", [(0, "A", "Temperature", 1)])
        summary, examples = self.audit(limit=0)
        self.assertEqual(summary["duplicate_key_count"], 0)
        self.assertIsNone(summary["identical_value_duplicate_keys_percent"])
        self.assertIsNone(summary["conflicting_value_difference_max"])
        self.assertIsNone(summary["conflicting_value_difference_mean"])
        self.assertIsNone(summary["primarily_cross_file_by_key"])
        self.assertEqual(examples, [])

    def test_empty_temperature_population(self):
        """
        Check that files containing no Temperature rows produce an empty temperature
        population, no examples, and an undefined duplicate-row percentage.
        """
        self.write_source("raw.csv", [(0, "A", "Humidity", 1)])
        summary, examples = self.audit()
        self.assertEqual(summary["rows_temperature"], 0)
        self.assertEqual(summary["unique_keys"], 0)
        self.assertIsNone(summary["duplicate_key_rate_percent"])
        self.assertEqual(examples, [])

    def test_identical_cross_file_values_and_no_normalization_of_labels(self):
        """
        Check exact parsed-float equality across files while leading spaces or other
        non-identical Sensor/DeviceId labels remain distinct rather than normalized.
        """
        self.write_source("a.csv", [
            (0, "A", "Temperature", "1.0"),
            (0, " A", "Temperature", 2),
            (0, "A", " Temperature", 3),
        ])
        self.write_source("b.csv", [(0, "A", "Temperature", "1e0")])
        summary, examples = self.audit()
        self.assertEqual(summary["rows_temperature"], 3)
        self.assertEqual(summary["unique_keys"], 2)
        self.assertEqual(summary["identical_value_duplicate_keys_percent"], 100)
        self.assertEqual(summary["conflicting_value_duplicate_keys_percent"], 0)
        self.assertEqual(summary["cross_file_identical_value_duplicate_keys"], 1)
        self.assertTrue(summary["primarily_cross_file_by_excess_rows"])
        self.assertIsNone(summary["conflicting_value_difference_mean"])
        self.assertEqual(examples, [])

    def test_cli_writes_only_two_isolated_reports_from_another_working_directory(self):
        """
        Invoke the audit CLI from a different working directory and check that
        aggregate-inclusive mode writes only its summary JSON and conflict CSV in
        the explicitly selected report directory.
        """
        source = self.write_source("raw.csv", [
            (0, "A", "Temperature", 1), (0, "A", "Temperature", 2),
        ])
        original = source.read_bytes()
        script = REPOSITORY / "scripts" / "phase1_implement.ipynb"
        completed = subprocess.run(
            [
                sys.executable, "-B", "-c",
                "import sys; from pathlib import Path; "
                f"sys.path.insert(0, {str(REPOSITORY)!r}); "
                "from scripts.shared_monitoring_engine import load_phase1_definitions; "
                f"load_phase1_definitions(Path({str(REPOSITORY)!r}), audit=True).main()",
                "--data-dir", str(self.data), "--out-dir", str(self.output),
                "--max-examples", "0",
            ],
            cwd=self.root, capture_output=True, text=True, check=True,
        )
        self.assertIn('"conflicting_value_duplicate_keys": 1', completed.stdout)
        self.assertEqual(
            {path.name for path in self.output.iterdir()},
            {"qc_phase1_duplicate_conflicts.csv", "qc_phase1_duplicate_summary.json"},
        )
        self.assertEqual(source.read_bytes(), original)
        with (self.output / "qc_phase1_duplicate_conflicts.csv").open(newline="") as stream:
            self.assertEqual(list(csv.DictReader(stream)), [])

    def test_example_provenance_preserves_exact_float_precision(self):
        """
        Check that conflict examples retain the exact parsed floating-point source
        values and row/file provenance without display rounding.
        """
        values = [32.735000610351555, 32.294998168945305]
        self.write_source(
            "raw.csv", [(0, "A", "Temperature", value) for value in values]
        )
        _, examples = self.audit()
        source = json.loads(examples[0]["source_files"])[0]
        self.assertEqual(source["values"], values)
        self.assertEqual(float(examples[0]["kept_value_phase1"]), values[0])
        self.assertEqual(float(examples[0]["min_value"]), min(values))
        self.assertEqual(float(examples[0]["max_value"]), max(values))

    def test_nonfinite_values_are_counted_and_strict_json_is_written(self):
        """
        Check non-finite temperature counting and ensure NaN/infinity values are
        represented as JSON-safe strings rather than non-standard JSON numeric
        tokens.
        """
        self.write_source("raw.csv", [
            (0, "A", "Temperature", "nan"),
            (0, "A", "Temperature", "nan"),
            (60, "A", "Temperature", "inf"),
            (60, "A", "Temperature", "inf"),
            (120, "A", "Temperature", "-inf"),
            (120, "A", "Temperature", "inf"),
            (180, "A", "Temperature", "-1e308"),
            (180, "A", "Temperature", "1e308"),
        ])
        summary, examples = self.audit()
        self.assertEqual(summary["rows_temperature_nonfinite"], 6)
        self.assertEqual(summary["identical_value_duplicate_keys"], 1)
        self.assertEqual(summary["conflicting_value_duplicate_keys"], 3)
        self.assertEqual(summary["conflicting_keys_with_undefined_difference"], 3)
        self.assertIsNone(summary["conflicting_value_difference_mean"])
        self.assertIsNone(summary["conflicting_value_difference_max"])
        self.assertTrue(all(row["abs_diff"] == "" for row in examples))
        self.assertEqual(json.loads(examples[0]["source_files"])[0]["values"], ["nan"])
        text = (self.output / "qc_phase1_duplicate_summary.json").read_text()
        json.loads(text, parse_constant=lambda value: self.fail(f"Invalid JSON: {value}"))

    def test_missing_sources_and_invalid_example_limit_fail_explicitly(self):
        """
        Check explicit errors for a missing data directory, no CSV sources, and a
        negative example limit, without creating the requested report directory.
        """
        with self.assertRaisesRegex(ValueError, "No source CSV"):
            run_qc(self.data, self.output)
        with self.assertRaisesRegex(ValueError, "does not exist"):
            run_qc(self.root / "missing", self.output)
        with self.assertRaisesRegex(ValueError, "non-negative"):
            run_qc(self.data, self.output, -1)
        self.assertFalse(self.output.exists())

    def gate_config(self):
        """
        Write a temporary YAML configuration containing the duplicate-key gate used
        only by source-exclusion audit fixtures.
        """
        config = self.root / "defaults.yaml"
        config.write_text(
            "quality_gates:\n  duplicate_key_rate_max_percent: 0.5\n",
            encoding="utf-8",
        )
        return config

    def audit_excluded(self):
        """
        Run source-only duplicate analysis with temperature_raw.csv excluded from
        those metrics, while retaining it as the aggregate for reconciliation
        against timestamped sources.
        """
        with contextlib.redirect_stderr(io.StringIO()):
            summary = run_qc(
                self.data, self.output, exclude_temperature_raw=True,
                config_path=self.gate_config(),
            )
        with (self.output / "qc_phase1_unmatched_records.csv").open(newline="") as stream:
            unmatched = list(csv.DictReader(stream))
        with (self.output / "qc_phase1_duplicate_summary.json").open() as stream:
            self.assertEqual(json.load(stream), summary)
        return summary, unmatched

    def test_exclusion_and_exact_multiset_match_preserve_multiplicity(self):
        """
        Check source-only duplicate metrics and exact aggregate/source
        record-multiset equality, including repeated identical records and their
        multiplicity.
        """
        self.write_source("temperature_raw.csv", [
            (0, "A", "Temperature", "1.0"),
            (0, "A", "Temperature", "1e0"),
            (60, "A", "Temperature", 2),
        ])
        self.write_source("2022_01_01_00-00-00_data.csv", [
            (0, "A", "Temperature", 1), (60, "A", "Temperature", 2),
        ])
        self.write_source(
            "2022_01_01_01-00-00.csv", [(0, "A", "Temperature", 1)]
        )
        self.write_source("notes.csv", [(180, "A", "Temperature", 9)])
        summary, unmatched = self.audit_excluded()
        self.assertEqual(summary["files_processed"], 3)
        self.assertEqual(summary["rows_temperature"], 4)
        self.assertEqual(summary["unique_keys"], 3)
        self.assertEqual(summary["duplicate_rows"], 1)
        self.assertEqual(summary["duplicate_key_count"], 1)
        self.assertEqual(summary["duplicate_multiplicity_distribution"], {"2": 1})
        self.assertEqual(summary["duplicate_key_rate_percent"], 25)
        self.assertFalse(summary["duplicate_row_gate"]["phase1_gate_passed"])
        self.assertEqual(
            summary["file_inventory"]["other_files_in_duplicate_statistics_only"],
            ["notes.csv"],
        )
        reconciliation = summary["aggregate_reconciliation"]
        self.assertEqual(reconciliation["aggregate_counts"]["rows_temperature"], 3)
        self.assertEqual(reconciliation["timestamped_source_counts"]["rows_temperature"], 3)
        self.assertEqual(reconciliation["matched_temperature_occurrences"], 3)
        self.assertTrue(reconciliation["valid_record_multisets_equal"])
        self.assertTrue(reconciliation["verification_complete"])
        self.assertTrue(reconciliation["all_temperature_records_match"])
        self.assertEqual(unmatched, [])

    def test_bidirectional_unmatched_counts_values_and_provenance(self):
        """
        Check that reconciliation reports excess records from either population with
        exact values, multiplicities, row numbers, and timestamped-source
        provenance.
        """
        aggregate = [
            (0, "A", "Temperature", 1), (0, "A", "Temperature", 1),
            (60, "A", "Temperature", "32.735000610351555"),
            (120, "A", "Temperature", 3),
        ]
        timestamped = [
            (0, "A", "Temperature", 1),
            (60, "A", "Temperature", "32.7350006103516"),
            (180, "A", "Temperature", 4),
        ]
        self.write_source("temperature_raw.csv", aggregate)
        source = self.write_source("2022_01_01_00-00-00_data.csv", timestamped)
        summary, unmatched = self.audit_excluded()
        reconciliation = summary["aggregate_reconciliation"]
        self.assertEqual(reconciliation["matched_temperature_occurrences"], 1)
        self.assertEqual(reconciliation["aggregate_unmatched_occurrences"], 3)
        self.assertEqual(reconciliation["timestamped_sources_unmatched_occurrences"], 2)
        self.assertEqual(
            reconciliation["unmatched_record_groups_by_direction"],
            {"aggregate_only": 3, "timestamped_sources_only": 2},
        )
        self.assertTrue(reconciliation["verification_complete"])
        self.assertFalse(reconciliation["valid_record_multisets_equal"])
        self.assertFalse(reconciliation["all_temperature_records_match"])
        self.assertEqual(len(unmatched), 5)
        multiplicity_mismatch = unmatched[0]
        self.assertEqual(multiplicity_mismatch["aggregate_count"], "2")
        self.assertEqual(multiplicity_mismatch["timestamped_sources_count"], "1")
        self.assertEqual(multiplicity_mismatch["unmatched_count"], "1")
        self.assertEqual(multiplicity_mismatch["aggregate_first_row_number"], "2")
        self.assertEqual(
            json.loads(multiplicity_mismatch["timestamped_source_files"]),
            [{"source_file": source.name, "occurrences": 1, "first_row_number": 2}],
        )
        tiny_conflicts = [row for row in unmatched if row["time_epoch"] == "60"]
        self.assertEqual(
            {float(row["value"]) for row in tiny_conflicts},
            {32.735000610351555, 32.7350006103516},
        )

    def test_spread_exceedances_and_bins_use_strict_boundaries(self):
        """
        Use values immediately below, exactly at, and immediately above each
        configured spread boundary to check strict exceedance counts and mutually
        exclusive bins.
        """
        spreads = [
            value for threshold in SPREAD_THRESHOLDS
            for value in (math.nextafter(threshold, 0), threshold, math.nextafter(threshold, math.inf))
        ]
        rows = [
            (index * 60, "A", "Temperature", value)
            for index, spread in enumerate(spreads) for value in (0, spread)
        ]
        self.write_source("temperature_raw.csv", rows)
        self.write_source("2022_01_01_00-00-00_data.csv", rows)
        summary, unmatched = self.audit_excluded()
        self.assertEqual(summary["duplicate_key_count"], len(spreads))
        self.assertEqual(summary["conflicting_value_duplicate_keys"], len(spreads))
        self.assertEqual(summary["duplicate_rows"], len(spreads))
        self.assertEqual(summary["duplicate_multiplicity_distribution"], {"2": len(spreads)})
        distribution = summary["conflicting_value_spread_distribution"]
        self.assertEqual(
            distribution["strictly_greater_than_counts"],
            {str(threshold): sum(value > threshold for value in spreads)
             for threshold in SPREAD_THRESHOLDS},
        )
        self.assertEqual(
            sum(row["keys"] for row in distribution["non_overlapping_bins"]),
            len(spreads),
        )
        for row in distribution["non_overlapping_bins"]:
            self.assertEqual(
                row["keys"],
                sum(value > row["lower_exclusive"]
                    and (row["upper_inclusive"] is None or value <= row["upper_inclusive"])
                    for value in spreads),
            )
        self.assertEqual(unmatched, [])

    def test_gate_distinguishes_strict_below_from_rounded_phase1_pass(self):
        """
        Check both the raw strict-below report and Phase 1 six-decimal-rounded <=
        gate, including rates just above the raw threshold that still pass after
        rounding.
        """
        config = self.gate_config()
        for rate, below, passed in (
            (0.1, True, True), (0.5, False, True),
            (0.5000004, False, True), (0.5000006, False, False),
        ):
            with self.subTest(rate=rate):
                result = duplicate_gate(rate, config)
                self.assertEqual(result["threshold_percent"], 0.5)
                self.assertEqual(result["strictly_below_threshold"], below)
                self.assertEqual(result["phase1_gate_passed"], passed)
                self.assertEqual(result["phase1_rounded_rate_percent"], round(rate, 6))
        self.assertIsNone(duplicate_gate(None, config)["phase1_gate_passed"])
        config.write_text("quality_gates:\n  duplicate_key_rate_max_percent: 0.1\n")
        self.assertFalse(duplicate_gate(0.2, config)["phase1_gate_passed"])
        for value in ("true", "missing", ".nan", "-1"):
            config.write_text(f"quality_gates:\n  duplicate_key_rate_max_percent: {value}\n")
            with self.assertRaises(ValueError):
                duplicate_gate(0, config)

    def test_invalid_and_nan_rows_cannot_produce_a_complete_match_claim(self):
        """
        Check that invalid-schema/temperature rows and NaN values keep record
        reconciliation from making an unqualified complete-match claim.
        """
        rows = [
            (0, "A", "Temperature", "nan"),
            (60, "A", "Temperature", "inf"),
            ("bad", "A", "Temperature", 1),
            (120, "A", "Temperature", "bad"),
        ]
        self.write_source("temperature_raw.csv", rows)
        self.write_source("2022_01_01_00-00-00_data.csv", rows)
        summary, unmatched = self.audit_excluded()
        reconciliation = summary["aggregate_reconciliation"]
        self.assertFalse(reconciliation["verification_complete"])
        self.assertFalse(reconciliation["all_temperature_records_match"])
        self.assertEqual(reconciliation["aggregate_unmatched_occurrences"], 1)
        self.assertEqual(reconciliation["timestamped_sources_unmatched_occurrences"], 1)
        self.assertEqual(reconciliation["matched_temperature_occurrences"], 1)
        for counts in (
            reconciliation["aggregate_counts"], reconciliation["timestamped_source_counts"],
        ):
            self.assertEqual(counts["rows_temperature_invalid"], 2)
            self.assertEqual(counts["rows_temperature_nonfinite"], 2)
        self.assertEqual(len(unmatched), 2)
        self.assertTrue(all(row["value"] == "nan" for row in unmatched))

    def test_exclusion_empty_population_does_not_claim_gate_pass(self):
        """
        Check that an empty source-only temperature population has no duplicate-gate
        pass result and no finite spread exceedances.
        """
        rows = [(0, "A", "Humidity", 1)]
        self.write_source("temperature_raw.csv", rows)
        self.write_source("2022_01_01_00-00-00_data.csv", rows)
        summary, unmatched = self.audit_excluded()
        self.assertIsNone(summary["duplicate_row_gate"]["phase1_gate_passed"])
        self.assertEqual(
            sum(summary["conflicting_value_spread_distribution"]["strictly_greater_than_counts"].values()),
            0,
        )
        self.assertTrue(summary["aggregate_reconciliation"]["all_temperature_records_match"])
        self.assertEqual(unmatched, [])

    def test_exclusion_requires_aggregate_and_timestamped_sources(self):
        """
        Check that source-exclusion mode refuses to run when the aggregate is
        missing or no timestamped source file is available.
        """
        self.write_source("2022_01_01_00-00-00_data.csv", [])
        with self.assertRaisesRegex(ValueError, "Missing aggregate"):
            run_qc(self.data, self.output, exclude_temperature_raw=True)
        self.write_source("temperature_raw.csv", [])
        (self.data / "2022_01_01_00-00-00_data.csv").unlink()
        self.write_source("2022_99_99_00-00-00.csv", [])
        with self.assertRaisesRegex(ValueError, "No timestamped"):
            run_qc(self.data, self.output, exclude_temperature_raw=True)
        self.assertFalse(self.output.exists())

    def test_exclusion_cli_preserves_original_report_directory(self):
        """
        Run the exclusion-mode CLI into a separate subdirectory and verify it writes
        three mode-specific reports without replacing an existing
        aggregate-inclusive report.
        """
        rows = [(0, "A", "Temperature", 1)]
        self.write_source("temperature_raw.csv", rows)
        self.write_source("2022_01_01_00-00-00_data.csv", rows)
        self.output.mkdir()
        sentinel = self.output / "qc_phase1_duplicate_summary.json"
        sentinel.write_text("original audit")
        excluded_output = self.output / "excluding_temperature_raw"
        script = REPOSITORY / "scripts" / "phase1_implement.ipynb"
        completed = subprocess.run(
            [
                sys.executable, "-B", "-c",
                "import sys; from pathlib import Path; "
                f"sys.path.insert(0, {str(REPOSITORY)!r}); "
                "from scripts.shared_monitoring_engine import load_phase1_definitions; "
                f"load_phase1_definitions(Path({str(REPOSITORY)!r}), audit=True).main()", "--exclude-temperature-raw",
                "--data-dir", str(self.data), "--out-dir", str(excluded_output),
                "--config", str(self.gate_config()),
            ],
            cwd=self.root, capture_output=True, text=True, check=True,
        )
        self.assertIn('"all_temperature_records_match": true', completed.stdout)
        self.assertEqual(sentinel.read_text(), "original audit")
        self.assertEqual(
            {path.name for path in excluded_output.iterdir()},
            {"qc_phase1_duplicate_conflicts.csv", "qc_phase1_duplicate_summary.json",
             "qc_phase1_unmatched_records.csv"},
        )


## Independent scientific and saved-artifact checks

The six local checks have different evidence strength. A synthetic alignment fixture checks minute bucketing, median combination within a bucket, missing-device blanks, confidence/coverage fields, and individual gate outcomes. Saved Phase 1 checks bind the passed producer record to implementation/configuration and output hashes, the expected schema, 218148 aligned buckets, 3471 source-inventory rows, general statistics, QC, and recorded dataset digests. Inventory paths are normalized to basenames before hashing to avoid workspace-root differences; these fixed values identify a recorded dataset revision, not universal constants.

The downstream check derives the six-sensor P90 with `prepare_monitoring`, validates current Phase 2/3 receipts and outputs, checks saved scientific CSV digests, and reconstructs the exact unrounded 19984-row by 82-column Phase 2 trace. It does not rerun the Phase 3 clustering grid. The raw-input check hashes every file under `Data/files_csv/` against a fixed identity, which can be I/O intensive. Negative receipt fixtures are in memory. Fresh-process checks load tagged definitions twice, reproduce a tiny synthetic transformation, and check that definition loading creates no extra files; they are not a full production run.

The fixed `BASELINE_REFERENCE_SHA256` identifies the preserved scientific reference seal; this notebook does not open a historical archive or rerun that original pipeline. Identity/digest comparisons confirm the specified saved bytes, not a general proof for every future dataset.

In [ ]:
import csv, gzip, io, subprocess
from scripts.shared_monitoring_engine import prepare_monitoring, load_validated, build_phase2_trace
from scripts.phase3_implement import _read_locked_observations, _validate_contract, _read_validation_record, load_phase3_analysis_results
import scripts.test_shared_monitoring as monitoring_tests

BASELINE_REFERENCE_SHA256 = "b4a2193e7528ddcf7a83f034bb91c0f3454dadfc27b92e353b6a89bf7a0501b2"
PUBLICATION = ROOT / "artifacts" / "phase1_operational" / "phase1_run.json"
INPUTS = [
    "scripts/phase1_test.ipynb", "scripts/phase1_implement.ipynb",
    "scripts/shared_monitoring_engine.py", "scripts/test_shared_monitoring.py",
    "scripts/phase3_implement.py", "scripts/phase3_test.py",
    "config/project_config.yaml",
    "artifacts/phase1_operational/phase1_run.json",
    "artifacts/phase1_operational/phase1_aligned_temperature.csv",
    "artifacts/phase1_operational/phase1_source_inventory.csv",
    "artifacts/phase2_outputs/phase2_summary.json",
    "artifacts/phase2_outputs/phase2_matched_trace.csv",
    "artifacts/phase2_operational/phase2_events.csv",
    "artifacts/phase2_operational/phase2_validation.json",
    "artifacts/phase3_outputs/phase3_results.csv",
    "artifacts/phase3_outputs/phase3_summary.json",
    "artifacts/phase3_outputs/phase3_cluster_sizes.csv",
    "artifacts/phase3_outputs/phase3_snapshots.csv",
    "artifacts/phase3_operational/phase3_equivalence_events.csv",
    "artifacts/phase3_operational/phase3_run.json",
    "artifacts/phase3_operational/phase3_validation.json",
]

def file_hash(path):
    """
    Return the SHA-256 digest of the exact bytes at path for identity and freshness
    checks.
    """
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def hashes():
    """
    Hash every path in INPUTS and a sorted inventory of all files below
    Data/files_csv; return the relative-path keyed tree digest as well. Raise if the
    raw input tree is missing or empty. This full-tree traversal is intentionally
    more expensive than hashing a single artifact.
    """
    result = {name: file_hash(ROOT / name) for name in INPUTS}
    raw = {path.relative_to(ROOT).as_posix(): file_hash(path)
           for path in sorted((ROOT / "Data" / "files_csv").rglob("*")) if path.is_file()}
    if not raw:
        raise ValueError("Phase 1 raw input directory is empty or missing.")
    result["raw_input_tree_sha256"] = hashlib.sha256(
        json.dumps(raw, sort_keys=True, separators=(",", ":")).encode("utf-8")
    ).hexdigest()
    return result

def validate_receipt(record, current, baseline_hash):
    """
    Raise ValueError unless the record is a complete passed schema-version-1 receipt
    with a positive test count, matching test IDs, no failures/errors/skips, the
    required check and not_run lists, current tested hashes, and the expected
    baseline identity.
    """
    count = record.get("test_count")
    if (record.get("schema_version") != 1 or record.get("status") != "passed"
        or not isinstance(count, int) or isinstance(count, bool) or count <= 0
        or not isinstance(record.get("test_ids"), list) or len(record["test_ids"]) != count
        or any(record.get(key) != [] for key in ("failures", "errors", "skipped"))
        or record.get("checks") != ["fixture_regressions", "saved_artifact_contract", "recorded_baseline_comparison",
                                    "full_precision_downstream_comparison", "fresh_process_reproduction", "no_write_definition_load"]
        or record.get("not_run") != ["exhaustive_raw_duplicate_audit"]
        or record.get("tested_sha256") != current or record.get("baseline_reference_sha256") != baseline_hash):
        raise ValueError("Phase 1 independent receipt is missing, failed, incomplete, or stale.")

def rows(path):
    """
    Read a CSV and return its header field names and all rows as dictionaries.
    """
    with path.open(encoding="utf-8", newline="") as stream:
        reader = csv.DictReader(stream)
        return reader.fieldnames, list(reader)

class Phase1IndependentEvidenceTests(unittest.TestCase):
    """
    Check a small synthetic alignment fixture and saved Phase 1/downstream evidence.
    These tests combine behavioral fixtures, artifact identity/regression
    comparisons, and isolated-process checks; none alone is a full production rerun.
    """
    def test_alignment_missing_confidence_and_gates(self):
        """
        Check UTC minute bucketing, median aggregation for multiple values of one
        device in a minute, blank missing-device cells, confidence and
        minimum-coverage fields, the coverage-derived stale_snapshot_rate, and each
        individual gate result on a synthetic fixture.
        """
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "fixture.csv"
            path.write_text("Time,DeviceId,Sensor,Value\n0,A,Temperature,10\n30,A,Temperature,14\n0,B,Temperature,20\n60,A,Temperature,30\n", encoding="utf-8")
            _, result, qc = phase1.build_phase1_temperature([path], 60, 120, 2, True, 0.7, 0.3)
            self.assertEqual([row["bucket_epoch"] for row in result], [0, 60])
            self.assertEqual((result[0]["A_temperature"], result[0]["B_temperature"]), (12.0, 20.0))
            self.assertEqual((result[1]["A_temperature"], result[1]["B_temperature"]), (30.0, ""))
            self.assertEqual([row["snapshot_confidence"] for row in result], [1.0, 0.65])
            self.assertEqual([row["meets_min_device_coverage"] for row in result], [1, 0])
            self.assertEqual(qc["stale_snapshot_rate_percent"], 50.0)
            passed, checks = phase1.evaluate_quality_gates(qc, {
                "schema_validity_min_percent": 100, "duplicate_key_rate_max_percent": 0.5,
                "stale_snapshot_rate_max_percent": 10, "device_coverage_min_percent": 90})
            self.assertFalse(passed)
            self.assertEqual([item["passed"] for item in checks], [True, True, False, False])

    def test_saved_phase1_contract_and_baseline(self):
        """
        Check passed producer status, configuration and implementation hashes,
        aligned/inventory hashes and schemas, the recorded 218148 aligned buckets
        and 3471 inventory rows, general-statistics/QC expectations, and fixed
        saved-data digests. Source paths are replaced with basenames before
        inventory hashing so workspace-specific absolute roots do not change the
        recorded identity; these values identify one dataset revision, not every
        future input.
        """
        status = json.loads(PUBLICATION.read_text(encoding="utf-8"))
        self.assertEqual((status["schema_version"], status["status"], status["limited_run"], status["quality_gates_passed"]), (1, "passed", False, True))
        self.assertEqual(status["implementation_sha256"], file_hash(ROOT / "scripts" / "phase1_implement.ipynb"))
        self.assertEqual(status["config_sha256"], file_hash(ROOT / "config" / "project_config.yaml"))
        operational = ROOT / "artifacts" / "phase1_operational"
        for key, name in (("raw_index", "phase1_source_inventory.csv"), ("phase1_aligned", "phase1_aligned_temperature.csv")):
            self.assertEqual(status["hashes"][key + "_sha256"], file_hash(operational / name))
        self.assertEqual(status["hashes"]["phase1_aligned_sha256"], "07f4eb49687e7f50e8d9179914e1f7b8a93ef83cef99b43cf9f073881051cb9b")
        columns, aligned = rows(operational / "phase1_aligned_temperature.csv")
        self.assertEqual(len(aligned), 218148)
        self.assertEqual(columns, ["bucket_epoch", "bucket_time_utc"] + [f"{c}_temperature" for c in "ABCDEFGH"] +
                         ["available_devices", "max_age_seconds", "snapshot_confidence", "meets_min_device_coverage",
                          "average_temperature_all_devices", "average_temperature_all_devices_count"])
        _, index = rows(operational / "phase1_source_inventory.csv")
        self.assertEqual(len(index), 3471)
        for row in index:
            row["file_path"] = Path(row["file_path"]).name
        normalized = hashlib.sha256(json.dumps(index, sort_keys=True, separators=(",", ":")).encode()).hexdigest()
        self.assertEqual(normalized, "2b0f18bb4654f3212da387c1bd6a7b9042dc824dfd29bd8ffea4b2438c22cbd4")
        self.assertEqual({key: str(value) for key, value in status["general_statistics"][0].items()}, {'series_name': 'all_devices_observed', 'source_column': 'average_temperature_all_devices', 'confidence_filter': 'snapshot_confidence_gt_0.75', 'filtered_snapshot_count': '216029', 'non_null_value_count': '216029', 'mean': '22.530749', 'std_dev': '1.488073', 'p05': '20.246875', 'p10': '20.7525', 'p15': '21.1075', 'p20': '21.381875', 'p25': '21.584286', 'p30': '21.78375', 'p35': '21.947143', 'p40': '22.108571', 'p45': '22.272143', 'p50': '22.43825', 'p55': '22.59125', 'p60': '22.768572', 'p65': '22.965625', 'p70': '23.161429', 'p75': '23.352143', 'p80': '23.578125', 'p85': '23.890714', 'p90': '24.343381', 'p95': '25.045'})
        self.assertEqual(status["qc"]["rows_temperature"], 9909343)
        self.assertEqual(status["qc"]["rows_temperature_duplicate"], 10090)
        self.assertEqual(status["qc"]["excluded_non_source_csv_filenames"], ["temperature_raw.csv"])
        self.assertTrue(all(gate["passed"] is True for gate in status["gate_checks"]))

    def test_downstream_full_precision_and_baseline(self):
        """
        Derive the six-sensor P90 through prepare_monitoring, validate current Phase
        2/3 evidence, compare recorded scientific CSV digests, and rebuild the
        unrounded 19984-by-82 Phase 2 trace for exact digest comparison. This does
        not rerun the Phase 3 clustering grid.
        """
        preparation = prepare_monitoring(ROOT)
        self.assertEqual(preparation["p90_value"], 22.428666649999997)
        self.assertEqual(preparation["statistics"]["non_null_value_count"], 19984)
        summary = load_validated(ROOT)
        self.assertEqual(summary["monitoring_preparation"], preparation)
        _read_validation_record(ROOT, ROOT / "artifacts" / "phase3_operational" / "phase3_validation.json")
        load_phase3_analysis_results(ROOT, ROOT / "artifacts" / "phase3_outputs")
        expected = {
            "phase2_outputs/phase2_matched_trace.csv": "ba261267a7751c2b044408f88cfa789c57e881b8f7102039acfe178fe5e89aeb",
            "phase2_operational/phase2_events.csv": "446d2e0b552ba603a0815e3275f5eba37513ac3298a63fb5129029dc3fba1d2e",
            "phase3_outputs/phase3_results.csv": "8bfe508355fd1de0254a5f4f17c3b65ca4e94b57ed99d7f16c79bafe9d3b9bef",
            "phase3_outputs/phase3_cluster_sizes.csv": "2f0cf6edb96904dd986d2345d610cb481b6a8ea0c7761158638786ba159b4ac2",
            "phase3_outputs/phase3_snapshots.csv": "b9c2c65b0ebb25ba960e8cf668f141b08e10b4bbd618e0a0da1023b19a00ac6c",
            "phase3_operational/phase3_equivalence_events.csv": "ff4a89ecec934a96456eeefa5e154b580b6ddac2251ef2711250380f2d489fcf",
        }
        for name, digest in expected.items():
            self.assertEqual(file_hash(ROOT / "artifacts" / name), digest, name)
        frame, columns = _read_locked_observations(_validate_contract(ROOT))
        trace = build_phase2_trace(frame, columns, preparation["p90_value"], round_output=False)
        self.assertEqual(trace.shape, (19984, 82))
        trace_bytes = trace.to_csv(index=False, lineterminator="\n", na_rep="").encode("utf-8")
        digest = hashlib.sha256(trace_bytes).hexdigest()
        self.assertEqual(digest, "ba261267a7751c2b044408f88cfa789c57e881b8f7102039acfe178fe5e89aeb")
        trace_path = ROOT / "artifacts" / "phase2_outputs" / "phase2_matched_trace.csv"
        self.assertEqual(trace_path.read_bytes(), trace_bytes)
        self.assertEqual(summary["trace_sha256"], digest)
        self.assertEqual(summary["trace_row_count"], len(trace))
        self.assertEqual(summary["trace_columns"], list(trace.columns))

    def test_raw_input_inventory(self):
        """
        Hash the complete raw CSV input tree and compare it with its recorded
        reference digest; this detects changed inputs but requires reading every
        file.
        """
        self.assertEqual(hashes()["raw_input_tree_sha256"], "3573ba3114361158776fee6879338449540a69ea8686c97f90aedb2926a0683d")

    def test_receipt_rejects_failure_missing_and_stale_evidence(self):
        """
        Use in-memory fabricated records to check rejection of absent, failed,
        empty, skipped, tampered, or baseline-stale validation evidence without
        writing the live receipt.
        """
        valid = {"schema_version": 1, "status": "passed", "test_count": 1, "test_ids": ["fixture"],
                 "failures": [], "errors": [], "skipped": [],
                 "checks": ["fixture_regressions", "saved_artifact_contract", "recorded_baseline_comparison",
                            "full_precision_downstream_comparison", "fresh_process_reproduction", "no_write_definition_load"],
                 "not_run": ["exhaustive_raw_duplicate_audit"], "tested_sha256": {"input": "a"},
                 "baseline_reference_sha256": "seal"}
        validate_receipt(valid, {"input": "a"}, "seal")
        for changed in ({}, {**valid, "status": "failed"}, {**valid, "test_ids": []},
                        {**valid, "skipped": ["fixture"]}, {**valid, "tested_sha256": {"input": "corrupt"}},
                        {**valid, "baseline_reference_sha256": "stale"}):
            with self.subTest(changed=changed), self.assertRaisesRegex(ValueError, "receipt"):
                validate_receipt(changed, {"input": "a"}, "seal")

    def test_fresh_process_reproducibility_and_no_write(self):
        """
        Load tagged core and audit definitions in two fresh Python processes, run
        the same tiny synthetic transform, and compare deterministic output. The
        fixture checks that definition loading adds no files; it is not a real-data
        rerun or proof that invoking the producer has no writes.
        """
        code = """import json, tempfile
from pathlib import Path
from scripts.shared_monitoring_engine import load_phase1_definitions
with tempfile.TemporaryDirectory() as temp:
    root = Path(temp)
    (root / 'scripts').mkdir()
    (root / 'scripts' / 'phase1_implement.ipynb').write_bytes(Path('scripts/phase1_implement.ipynb').read_bytes())
    before = sorted(str(p.relative_to(root)) for p in root.rglob('*'))
    core = load_phase1_definitions(root)
    audit = load_phase1_definitions(root, audit=True)
    source = root / 'fixture.csv'
    source.write_text('Time,DeviceId,Sensor,Value\\n0,A,Temperature,10\\n0,A,Temperature,99\\n60,A,Temperature,20\\n')
    _, aligned, qc = core.build_phase1_temperature([source], 60, 120, 1, True, .7, .3)
    after = sorted(str(p.relative_to(root)) for p in root.rglob('*') if p != source)
    assert before == after and callable(audit.run_qc)
    print(json.dumps({'aligned': aligned, 'qc': qc}, sort_keys=True, allow_nan=False))
"""
        first = subprocess.run([sys.executable, "-B", "-c", code], cwd=ROOT, capture_output=True, check=True).stdout
        second = subprocess.run([sys.executable, "-B", "-c", code], cwd=ROOT, capture_output=True, check=True).stdout
        self.assertEqual(first, second)
        value = json.loads(first)
        self.assertEqual([row["A_temperature"] for row in value["aligned"]], [10.0, 20.0])
        self.assertEqual(value["qc"]["rows_temperature_duplicate"], 1)


## Execute checks and publish the independent receipt

The final cell combines all three local test classes (28 local methods in this notebook) with the imported `scripts.test_shared_monitoring` suite. The total is derived from the loaded suite, not a fixed count. It rejects failures, errors, skipped tests, an empty/incomplete run, or any change to a hashed input during execution. `hashes()` covers the test and implementation notebooks, shared and Phase 3 code, project configuration, the Phase 1 producer record, current Phase 2/3 validation receipts, and required artifacts, and the complete raw-input-tree digest.

A passed `phase1_validation.json` records `test_ids`, `test_count`, `failures`, `errors`, `skipped`, `checks`, `not_run`, `tested_sha256`, and `baseline_reference_sha256`. `not_run: ["exhaustive_raw_duplicate_audit"]` explicitly records that the expensive full duplicate investigation is omitted. The final receipt is atomically published only after those checks pass; on a caught exception the runner atomically writes failed status and re-raises. A previous receipt is not current when any tested hash or the reference identity changes. Producer status and Phase 2/3 receipts are evidence inputs; this notebook owns only `phase1_validation.json`.

In [ ]:
try:
    # Snapshot all source, evidence, and raw-tree identities before executing tests.
    before = hashes()
    suite = unittest.TestSuite([
        unittest.defaultTestLoader.loadTestsFromTestCase(Phase1SourceDiscoveryTests),
        unittest.defaultTestLoader.loadTestsFromTestCase(DuplicateQCTests),
        unittest.defaultTestLoader.loadTestsFromTestCase(Phase1IndependentEvidenceTests),
        unittest.defaultTestLoader.loadTestsFromModule(monitoring_tests),
    ])
    def ids(suite):
        """
        Yield each unittest test ID recursively from a nested TestSuite so the
        receipt records the exact suite contents without maintaining a hard-coded
        total.
        """
        for item in suite:
            if isinstance(item, unittest.TestSuite):
                yield from ids(item)
            else:
                yield item.id()
    test_ids = list(ids(suite))
    result = unittest.TextTestRunner(verbosity=2).run(suite)
    if not result.wasSuccessful() or result.skipped or result.testsRun != len(test_ids) or not test_ids:
        raise RuntimeError(f"Independent Phase 1 tests failed or skipped: failures={len(result.failures)}, errors={len(result.errors)}, skipped={len(result.skipped)}")
    # A test cannot pass if any hashed input changes while the suite is running.
    after = hashes()
    if before != after:
        raise RuntimeError("Inputs, source or saved scientific artifacts changed during validation.")
    receipt = {
        "schema_version": 1, "status": "passed",
        "test_count": result.testsRun, "test_ids": test_ids, "failures": [], "errors": [], "skipped": [],
        "checks": ["fixture_regressions", "saved_artifact_contract", "recorded_baseline_comparison",
                   "full_precision_downstream_comparison", "fresh_process_reproduction", "no_write_definition_load"],
        "not_run": ["exhaustive_raw_duplicate_audit"],
        "baseline_reference_sha256": BASELINE_REFERENCE_SHA256,
        "tested_sha256": after,
    }
    validate_receipt(receipt, after, BASELINE_REFERENCE_SHA256)
    atomic_receipt(receipt)
    print(f"Independent Phase 1 validation passed: {result.testsRun} tests; receipt: {RECEIPT}")
except Exception as error:
    atomic_receipt({"schema_version": 1, "status": "failed", "error_type": type(error).__name__, "error": str(error)})
    raise
